# 01 - Data Collection
Collect real ETF and benchmark data using the project data loader.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

# Ensure project modules are importable when running from notebooks/.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Install any missing dependencies before importing project modules.
for package in ["yfinance", "plotly"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from config import PAIR_CONFIGS
from src.data_loader import MarketDataLoader

loader = MarketDataLoader()
panel = loader.fetch_universe(PAIR_CONFIGS, period="2y", interval="1d")
print(f"Universe panel: {panel.shape[0]:,} rows × {panel.shape[1]} columns")
panel.head()

,etf_close,etf_high,etf_low,etf_open,etf_volume,benchmark_close,benchmark_high,benchmark_low,benchmark_open,benchmark_volume,pair,etf_ticker,benchmark_ticker
Date,,,,,,,,,,,,,
2024-04-10 00:00:00+00:00,501.905151,503.896662,499.923413,501.280343,82652800,5160.640137,5178.430176,5138.700195,5167.879883,3845930000,SPY_^GSPC,SPY,^GSPC
2024-04-10 00:00:00+00:00,48.486691,48.732099,48.222408,48.297915,1897900,5000.830078,5037.180176,4951.189941,5006.500000,28802000,FEZ_^STOXX50E,FEZ,^STOXX50E
2024-04-10 00:00:00+00:00,433.607971,434.468516,431.540679,432.252858,61502200,18011.660156,18040.830078,17932.419922,17957.960938,5308250000,QQQ_^NDX,QQQ,^NDX
2024-04-10 00:00:00+00:00,137.942795,138.534367,137.525778,137.913702,161300,104.793976,105.209787,104.378165,104.706942,7321600,URTH_ACWI,URTH,ACWI
2024-04-11 00:00:00+00:00,440.531830,441.481393,433.202323,435.477356,45474600,18307.980469,18337.150391,17998.250000,18085.109375,4714750000,QQQ_^NDX,QQQ,^NDX


In [ ]:
# --- Per-pair coverage summary --------------------------------------------------
coverage = (
    panel.groupby("pair")
    .agg(
        etf=("etf_ticker", "first"),
        benchmark=("benchmark_ticker", "first"),
        start=("etf_close", lambda s: s.index.min()),
        end=("etf_close", lambda s: s.index.max()),
        rows=("etf_close", "count"),
        missing_etf=("etf_close", lambda s: s.isna().sum()),
        missing_bench=("benchmark_close", lambda s: s.isna().sum()),
    )
    .reset_index()
)
coverage["trading_days"] = (pd.to_datetime(coverage["end"]) - pd.to_datetime(coverage["start"])).dt.days
coverage

,pair,etf_close,benchmark_close
Date,,,
2026-04-10 00:00:00+00:00,SPY_^GSPC,679.940002,6822.919922
2026-04-10 00:00:00+00:00,QQQ_^NDX,611.260071,25127.449219
2026-04-10 00:00:00+00:00,FEZ_^STOXX50E,66.019997,5926.109863
2026-04-10 00:00:00+00:00,URTH_ACWI,188.460007,145.205002


In [ ]:
# --- Normalized close-price chart (base-100 at earliest observation) -----------
# Normalizing to 100 lets all pairs appear on the same axis regardless of price scale.
fig = go.Figure()
for pair_name, pair_df in panel.groupby("pair"):
    pair_df = pair_df.sort_index()
    etf_norm  = pair_df["etf_close"]  / pair_df["etf_close"].iloc[0]  * 100
    bench_norm = pair_df["benchmark_close"] / pair_df["benchmark_close"].iloc[0] * 100
    etf_ticker   = pair_df["etf_ticker"].iloc[0]
    bench_ticker = pair_df["benchmark_ticker"].iloc[0]

    fig.add_trace(go.Scatter(x=pair_df.index, y=etf_norm,    mode="lines", name=f"{etf_ticker}  (ETF)"))
    fig.add_trace(go.Scatter(x=pair_df.index, y=bench_norm,  mode="lines", name=f"{bench_ticker} (Benchmark)",
                             line=dict(dash="dot")))

fig.update_layout(
    title="Normalized Close Prices – All Pairs (Base 100)",
    xaxis_title="Date",
    yaxis_title="Indexed Price (Base 100)",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

In [ ]:
# --- Intraday daily return spread (ETF return – Benchmark return per day) -------
# This gives a first visual hint at where tracking error concentrates.
panel_copy = panel.copy().sort_index()
panel_copy["etf_ret"]       = panel_copy.groupby("pair")["etf_close"].pct_change()
panel_copy["benchmark_ret"] = panel_copy.groupby("pair")["benchmark_close"].pct_change()
panel_copy["return_spread"] = panel_copy["etf_ret"] - panel_copy["benchmark_ret"]

box_data = panel_copy.dropna(subset=["return_spread"])
fig2 = px.box(
    box_data,
    x="pair",
    y="return_spread",
    color="pair",
    title="Daily Return Spread Distribution per Pair  (ETF return − Benchmark return)",
    labels={"return_spread": "Return Spread", "pair": "Pair"},
    template="plotly_white",
    height=420,
)
fig2.update_layout(showlegend=False)
fig2.show()